# Model search: Optuna with a model-size penalty, then strategy sims

Adapted from `buy_model_optuna.ipynb`. The two models are selected in turn, each first by its validation metric (with a size penalty), then by **strategy sims**:

| Step | What happens | Result |
|---|---|---|
| **A1** | Optuna search for the **move** model (+10% within 30 min) on train / val_train; metric partial AUC (FPR ≤ 0.10) | every non-pruned trial saved |
| **B1** | each of the top `TOP_N` move models **alone** in strategy sims (buy when the move score, as a tune percentile, ≥ the tuned confidence) | **the chosen move model** |
| **A2** | Optuna search for the **direction** model (+15% before −15% within 2 h), trained only on the **chosen move model's candidate minutes**: tradable, move score in its top 50%, outcome decided (the A50 recipe); metric AUC | every non-pruned trial saved |
| **B2** | each of the top `TOP_N` direction models **paired with the chosen move model**: combo = move × direction as a tune percentile | **the chosen direction model** |

- **Searches:** objective = **validation metric − `SIZE_PENALTY_ALPHA` × size in MB**; models over `MODEL_SIZE_CAP_MB` are excluded. Each non-pruned trial is saved as a `.ubj` plus `_params.json` (direction files: `dir_for_<move model>_trialNNN.*`).
- **Sims:** `RUNS_Y` strategy optimisations per model or pair, with rebase exits and the fixed, causality-tested simulator; optimised on **tune**, selected and reported on **val_tune**; run i uses the same optimizer seed everywhere. Defaults: `TOP_N = 10`, `RUNS_Y = 10`, so 200 strategy runs.
- **Choosing:** `PICK_RULE` (the best mean, or the smallest model within 2 standard errors of it), or by hand with `MOVE_CHOICE` / `DIR_CHOICE`.

**Final cell:** both stages' tables, the chosen pair, and **the value of the direction model**: every pair against the chosen move model alone, run by run with the same seeds.

**Input:** the split CSVs of `Dataset_creationV9` as they are (every feature, incl. the 22 window features, is computed there) and the cleaned OHLCV in `processed_data/`.

**Splits:** train / val_train (models) and tune / val_tune (sims). The **test** split is never read.

**Output:** every file goes into `model_search/`. **Run All.** Resumable.

In [1]:
# ============================================================================
# CONFIG — everything tunable lives here
# ============================================================================
# This notebook (adapted from buy_model_optuna.ipynb) selects the two models in turn:
#   A1  MOVE model (label: +10% within 30 min): Optuna search. Every NON-PRUNED trial's
#       booster is saved as a .ubj + a hyperparameter JSON.
#       Objective = maximize (val metric - SIZE_PENALTY_ALPHA * size_mb).
#   B1  each of the TOP_N move models ALONE in strategy sims (buy when the move score,
#       as a tune percentile, >= the tuned confidence), RUNS_Y runs each
#       -> the chosen move model
#   A2  DIRECTION model (label: +15% before -15% within 2 h): Optuna search, trained
#       only on the CHOSEN move model's candidate minutes (the A50 recipe: tradable 12 h
#       after listing, both price positions <= 0.575, move score >= its own train
#       median, outcome decided). Same objective; files dir_for_<move>_trialNNN.*
#   B2  each of the TOP_N direction models paired with the chosen move model (A50:
#       combo = move x direction as a tune percentile), RUNS_Y runs each
#       -> the chosen direction model
#   FINAL  both stages' tables, the chosen pair, and the VALUE OF THE DIRECTION MODEL:
#       every pair vs the chosen move model alone (same sampler seeds -> paired).
# All sims: rebase exits, the fixed (causality-tested) simulator; Optuna optimises
# TUNE, VAL_TUNE is sampled for early stopping and selection; run i uses the same
# sampler seed everywhere.
# Splits: train / val_train (models), tune / val_tune (sims). The TEST split is
# never read. Every file this notebook creates goes into OUT_DIR.
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import optuna.samplers
import json
import time
import gc
import warnings
from pathlib import Path
from numba import njit
from sklearn.metrics import roc_auc_score
from scipy import stats as sst

warnings.simplefilter('ignore', pd.errors.PerformanceWarning)

# ── INPUTS ───────────────────────────────────────────────────────────────────
V9_SPLITS = Path("dataset_v9") / "splits"       # train / val_train / tune / val_tune (test is never read)
OHLCV_FOLDER = "processed_data"                 # cleaned OHLCV (data_cleaning.ipynb)
MOVE_LABEL = 'label_10pct_30m'
DIR_LABEL = 'label_fpclass_15pct_2h'            # 1 = down first, 2 = up first (0 = undecided)
META_COL = 'label_meta_minutes_to_end'
MIN_MINUTES_TO_END = 240                        # move model rows (as A50)
DIR_SUFFIX = '_p15_2h'                          # the direction model's pump-feature set

# ── OUTPUT: ONE FOLDER ───────────────────────────────────────────────────────
OUT_DIR = Path("model_search")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── PHASE A: hyperparameter search (per model type) ──────────────────────────
HPO_TRIALS = {'move': 60, 'dir': 60}            # total Optuna trials per model type
N_STARTUP = 15                                  # random startup trials before TPE/pruning
PRUNE_WARMUP_ROUNDS = 250                       # no trial may be pruned before this many rounds
N_ESTIMATORS_MAX = 5000
EARLY_STOPPING_ROUNDS = 100
SEED = 1
SIZE_PENALTY_ALPHA = 0.001                      # objective score = metric - SIZE_PENALTY_ALPHA * size_mb
MODEL_SIZE_CAP_MB = 20                          # completed models above this are excluded from the top N
METRIC = {'move': 'pauc', 'dir': 'auc'}         # 'pauc' = partial AUC at FPR <= PARTIAL_AUC_MAX_FPR, 'auc' = AUC
PARTIAL_AUC_MAX_FPR = 0.10
CAND_FRAC = 0.50                                # direction candidates: move score in its top 50% (A50)
TRADING_START_MIN = 12 * 60
A50_PP_CAP = 0.575

PARAMS_FIXED = {
    'objective':    'binary:logistic',
    'eval_metric':  'auc',
    'tree_method':  'hist',
    'device':       'cuda',      # NVIDIA GPU
    'random_state': 1,
    'n_jobs':       -1,
}


def suggest_hparams(trial):
    return {
        'grow_policy':       trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 3e-1, log=True),
        'min_child_weight':  trial.suggest_float('min_child_weight', 10.0, 5000.0, log=True),
        'gamma':             trial.suggest_float('gamma', 1e-2, 50.0, log=True),
        'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.2, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.2, 1.0),
        'colsample_bynode':  trial.suggest_float('colsample_bynode', 0.2, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 50.0, log=True),
        'max_delta_step':    trial.suggest_float('max_delta_step', 0.0, 10.0),
    }


# ── PHASE B: sequential selection by strategy sims ──────────────────────────
TOP_N = 10                                      # best models per type (by Phase A score) that go to the sims
RUNS_Y = 10                                     # strategy optimisation runs per simulated pair
PICK_RULE = 'best_mean'                         # 'best_mean', or 'smallest_within_2se': the smallest model
                                                # whose mean is within 2 standard errors of the best mean
MOVE_CHOICE = None                              # None = automatic (PICK_RULE); or a model stem, e.g. 'move_trial017',
DIR_CHOICE = None                               # to override the choice of stage B1 / B2 by hand
                                                # (changing MOVE_CHOICE starts a new direction search for that move model)
SIM_MODES = ['rebase']
SCORE_CAPITAL_CEILING_X = 10
SCORE_LOG_FLOOR_X = 0.001
TRADING_START_HOURS = 12
SCORE_MODE = 'capped'
TUNE_TAKE_PROFIT = True
ACTIVATION_BOUNDS = (0.10, 0.35)
ACTIVATION_THRESHOLD = 0.15
CONF_BOUNDS = (0.0, 1.0)
FIXED_PARAMS = {}
base_config = {
    'sol_price_usd': 130.0, 'initial_capital_usd': 20, 'slippage_pct': 0.015,
    'desired_price_impact_pct': 0.00333, 'target_fill_pct': 1,
    'trading_start_minutes': TRADING_START_HOURS * 60, 'data_folder': OHLCV_FOLDER,
    'exit_mode': 'rebase',
    'confidence_threshold': 0.8, 'activation_threshold': ACTIVATION_THRESHOLD,
    'stale_trade_minutes': 750, 'max_price_position_15m': 0.575, 'max_price_position_60m': 0.575,
    'reentry_threshold_pct': 0.10, 'loss_floor_pct': 0.25, 'loss_floor_consecutive_minutes': 3,
    'reentry_decay_rate': 0.0029, 'reentry_decay_start_minutes': 71,
}
N_TRIALS, MAX_TRIALS = 500, 6000                # strategy optimisation: min / max trials per run
CMAES_STARTUP_TRIALS = 250
EARLY_STOP_WARMUP = CMAES_STARTUP_TRIALS
VAL_EVAL_EVERY = 25
EARLY_STOP_PATIENCE_SAMPLES = 8
EARLY_STOP_TOL_FRAC, EARLY_STOP_TOL_MIN, EARLY_STOP_TOL_MIN_SAMPLES = 0.10, 0.001, 5
OPTUNA_BASE_SEED = 1
BOOTSTRAP_N = 2000

print(f"Searches: {HPO_TRIALS['move']} move + {HPO_TRIALS['dir']} direction trials; score = metric - "
      f"{SIZE_PENALTY_ALPHA} * size_mb (metric: move {METRIC['move']}, direction {METRIC['dir']}); cap {MODEL_SIZE_CAP_MB} MB")
print(f"Sims: B1 {TOP_N} move models alone + B2 {TOP_N} direction models with the chosen move model, {RUNS_Y} runs each "
      f"({2 * TOP_N * RUNS_Y} strategy runs; optimise tune, select on val_tune; test is never read)  ->  {OUT_DIR}/")

Searches: 60 move + 60 direction trials; score = metric - 0.001 * size_mb (metric: move pauc, direction auc); cap 20 MB
Sims: B1 10 move models alone + B2 10 direction models with the chosen move model, 10 runs each (200 strategy runs; optimise tune, select on val_tune; test is never read)  ->  model_search/


In [2]:
# ============================================================================
# DATA — the V9 splits with the current feature set, cached in OUT_DIR
# ============================================================================
# The splits of Dataset_creationV9, as they are: every feature the models use is
# computed there (incl. the 22 window features). The OHLCV (for the tradable-minute
# check and the sims) is read from OHLCV_FOLDER. The move model uses every feature
# except the direction pump set (*_p15_2h); the direction model uses every feature
# except the move pump set (the same names without the suffix).
# TEST — the model splits share no coin with the sim splits; no label is a feature.
_KEYS = ['coin_id', 'timestamp']


def _read_ohlcv_raw(coin):
    f = list(Path(OHLCV_FOLDER).glob(f"{coin}_ohlcv_cleaned.csv")) or \
        list(Path(OHLCV_FOLDER).glob(f"{coin.replace('_cleaned', '')}_ohlcv_cleaned.csv"))
    if not f:
        return None
    df = pd.read_csv(f[0])
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True, errors='coerce')
    return df[df['timestamp'].notna()].sort_values('timestamp').reset_index(drop=True)


def load_split(name):
    """A V9 split (cached as parquet in OUT_DIR for speed)."""
    cp = OUT_DIR / f"cache_{name}.parquet"
    if cp.exists():
        return pd.read_parquet(cp)
    df = pd.read_csv(V9_SPLITS / f"{name}.csv")
    df['coin_id'] = df['coin_id'].astype(str)
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)
    if 'return_3d' not in df.columns:
        raise RuntimeError(f"{name}.csv has no window features — rebuild it with the current Dataset_creationV9")
    df.to_parquet(cp, index=False)
    return df


def ohlcv_positions(keys):
    """position of every row in its coin's OHLCV timeline (-1 if not found)."""
    pos = np.full(len(keys), -1, dtype=np.int64)
    for coin, idx in keys.groupby('coin_id', sort=False).indices.items():
        o = _read_ohlcv_raw(coin)
        if o is None:
            continue
        ts = o['timestamp'].to_numpy()
        q = keys['timestamp'].to_numpy()[idx]
        p = np.searchsorted(ts, q)
        ok = (p < len(ts)) & (ts[np.minimum(p, len(ts) - 1)] == q)
        pos[idx[ok]] = p[ok]
    return pos


def eligible_flags(df):
    """tradable minutes: 12 h after the coin's first minute, both price positions <= A50_PP_CAP."""
    pos = ohlcv_positions(df[_KEYS])
    return ((pos >= TRADING_START_MIN) & (df['price_position_15m'].to_numpy() <= A50_PP_CAP)
            & (df['price_position_1h'].to_numpy() <= A50_PP_CAP))


def feature_columns(df, kind):
    cols = [c for c in df.columns if c not in _KEYS and not c.startswith('label')]
    set2 = [c for c in cols if c.endswith(DIR_SUFFIX)]
    if kind == 'move':
        return [c for c in cols if c not in set2]
    set1 = {c[:-len(DIR_SUFFIX)] for c in set2}
    return [c for c in cols if c not in set1]


def load_model(path):
    m = xgb.Booster()
    m.load_model(str(path))
    return m


def predict(model, df):
    f = list(model.feature_names)
    return model.predict(xgb.DMatrix(df[f], feature_names=f), iteration_range=(0, model.best_iteration + 1))


def to_percentile(values, reference):
    ref = np.sort(np.asarray(reference, dtype=np.float64))
    return np.clip(np.searchsorted(ref, np.asarray(values, dtype=np.float64), side='right') / len(ref), 1e-6, 1.0)


def _coins(name):
    return set(pd.read_csv(V9_SPLITS / f"{name}.csv", usecols=['coin_id'])['coin_id'].astype(str).unique())


_model_coins, _sim_coins = _coins('train') | _coins('val_train'), _coins('tune') | _coins('val_tune')
if _model_coins & _sim_coins or _coins('train') & _coins('val_train') or _coins('tune') & _coins('val_tune'):
    raise RuntimeError("TEST failed: splits overlap")
print("✓ TEST: train / val_train / tune / val_tune share no coins (test is never read)")

train_df, val_df = load_split('train'), load_split('val_train')
train_df = train_df[train_df[META_COL] >= MIN_MINUTES_TO_END].reset_index(drop=True)
val_df = val_df[val_df[META_COL] >= MIN_MINUTES_TO_END].reset_index(drop=True)
FEATS = {k: feature_columns(train_df, k) for k in ('move', 'dir')}
for k, f in FEATS.items():
    assert not any(c.startswith('label') for c in f), f"a label is in the {k} features"
print(f"✓ TEST: no label among the features (move {len(FEATS['move'])}, direction {len(FEATS['dir'])} features)")
print(f"  train {len(train_df):,} rows, val_train {len(val_df):,} rows (minutes_to_end >= {MIN_MINUTES_TO_END})")

✓ TEST: train / val_train / tune / val_tune share no coins (test is never read)
✓ TEST: no label among the features (move 146, direction 146 features)
  train 988,626 rows, val_train 593,173 rows (minutes_to_end >= 240)


In [3]:
# ============================================================================
# PHASE A · the Optuna search (from buy_model_optuna.ipynb, for one model type)
# ============================================================================
# HPO_TRIALS[kind] trials · N_STARTUP random startup · MedianPruner (no pruning until
# a trial has trained PRUNE_WARMUP_ROUNDS boosting rounds). Study direction is
# MAXIMIZE (metric - alpha*size); the pruning signal is the validation AUC. The
# ranking metric is METRIC[kind] ('pauc' = partial AUC at FPR <= PARTIAL_AUC_MAX_FPR,
# McClish-normalised; 'auc' = AUC). Non-pruned trials save <kind>_trialNNN.ubj +
# <kind>_trialNNN_params.json in OUT_DIR. Pruned trials save nothing. Resumable.
PRUNE_REPORT_EVERY = 25


def _unique_trial_stem(prefix, n):
    """'<kind>_trialNNN', or with '_rK' appended if that file already exists (after
    resuming), so a NEW trial can NEVER overwrite a model saved by a PRIOR run."""
    stem = f"{prefix}_trial{n:03d}"
    if not (OUT_DIR / f"{stem}.ubj").exists():
        return stem
    k = 1
    while (OUT_DIR / f"{stem}_r{k}.ubj").exists():
        k += 1
    return f"{stem}_r{k}"


class OptunaPruningCallback(xgb.callback.TrainingCallback):
    """Reports validation AUC to Optuna (throttled) and prunes when told to."""
    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        if epoch < PRUNE_WARMUP_ROUNDS or (epoch % PRUNE_REPORT_EVERY) != 0:
            return False
        d = evals_log.get('validation_0') or next(iter(evals_log.values()))
        self.trial.report(d['auc'][-1], step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def run_hpo(kind, prefix, dtrain, deval, y_val, scale_pos_weight):
    """Run (or resume) the study for one model type ('move' / 'dir'; HPO_TRIALS / METRIC are
    looked up by kind). Files are named <prefix>_trialNNN.*; returns the ranked completed trials."""
    metric = METRIC[kind]

    def objective(trial):
        params = dict(PARAMS_FIXED, **suggest_hparams(trial))
        params['scale_pos_weight'] = scale_pos_weight          # fixed, data-derived
        es = xgb.callback.EarlyStopping(rounds=EARLY_STOPPING_ROUNDS, min_delta=1e-4,
                                        save_best=True, maximize=True, metric_name='auc')
        model = None
        try:
            model = xgb.train(params, dtrain, num_boost_round=N_ESTIMATORS_MAX,
                              evals=[(deval, 'validation_0')],
                              callbacks=[es, OptunaPruningCallback(trial)], verbose_eval=False)
            val_pred = model.predict(deval)
            auc = float(roc_auc_score(y_val, val_pred))
            pauc = float(roc_auc_score(y_val, val_pred, max_fpr=PARTIAL_AUC_MAX_FPR))
            value = pauc if metric == 'pauc' else auc
            best_it = int(model.best_iteration)

            # ── Persist model (.ubj) + hyperparameters (only because not pruned) ──
            stem = _unique_trial_stem(prefix, trial.number)
            model_path = OUT_DIR / f"{stem}.ubj"
            params_path = OUT_DIR / f"{stem}_params.json"
            model.save_model(str(model_path))
            size_mb = model_path.stat().st_size / (1024.0 * 1024.0)
            score = float(value) - SIZE_PENALTY_ALPHA * size_mb
            json.dump({
                'model_type':       kind,
                'trial_number':     trial.number,
                'score':            score,
                'metric':           metric,
                'pauc':             pauc,
                'partial_auc_max_fpr': PARTIAL_AUC_MAX_FPR,
                'auc':              auc,
                'size_mb':          size_mb,
                'best_iteration':   best_it,
                'scale_pos_weight': scale_pos_weight,
                'tuned_params':     {k: trial.params[k] for k in trial.params},
                'fixed_params':     PARAMS_FIXED,
                'model_file':       model_path.name,
            }, open(params_path, 'w'), indent=2, default=str)
            for k, v in (('pauc', pauc), ('auc', auc), ('size_mb', float(size_mb)), ('best_iteration', best_it),
                         ('model_file', model_path.name)):
                trial.set_user_attr(k, v)
            return score
        finally:
            del model
            gc.collect()

    def _progress_cb(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            info = (f"pAUC={trial.user_attrs.get('pauc', np.nan):.4f}  AUC={trial.user_attrs.get('auc', np.nan):.4f}  "
                    f"score={trial.value:.4f}  size={trial.user_attrs.get('size_mb', np.nan):6.1f}MB")
        else:
            info = "pruned"
        try:
            best_str = f"{study.best_value:.4f}"
        except ValueError:
            best_str = "  -  "
        print(f"  [{prefix} trial {trial.number:>3}/{HPO_TRIALS[kind]}] {trial.state.name:<8} {info}  |  best score={best_str}",
              flush=True)

    print("=" * 70 + f"\n{kind.upper()} MODEL HYPERPARAMETER SEARCH ({prefix}; metric {metric})\n" + "=" * 70)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    storage_path = OUT_DIR / f"{prefix}_study.log"
    lock = optuna.storages.journal.JournalFileOpenLock(str(storage_path))
    storage = optuna.storages.JournalStorage(optuna.storages.journal.JournalFileBackend(str(storage_path), lock_obj=lock))
    study = optuna.create_study(
        direction='maximize', study_name=f"{prefix}_model_hpo", storage=storage, load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=PRUNE_WARMUP_ROUNDS))

    # ── Bridge: fold in any saved-model trials NOT already represented in the study
    existing = {t.user_attrs.get('model_file') for t in study.trials if t.user_attrs.get('model_file')}
    for pj in sorted(OUT_DIR.glob(f'{prefix}_trial*_params.json')):
        try:
            meta = json.load(open(pj))
        except Exception:
            continue
        mf = meta.get('model_file')
        if (not mf) or (mf in existing) or not (OUT_DIR / mf).exists():
            continue
        study.add_trial(optuna.trial.create_trial(
            state=optuna.trial.TrialState.COMPLETE, value=float(meta.get('score')), params={}, distributions={},
            user_attrs={'pauc': meta.get('pauc'), 'auc': meta.get('auc'), 'size_mb': meta.get('size_mb'),
                        'best_iteration': meta.get('best_iteration'), 'model_file': mf, 'reconstructed': True}))
        existing.add(mf)

    remaining = max(0, HPO_TRIALS[kind] - len(study.trials))
    print(f"  {len(study.trials)} prior trial(s); running {remaining} more")
    t0 = time.time()
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, n_jobs=1, callbacks=[_progress_cb], show_progress_bar=False)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    under_cap = [t for t in completed if t.user_attrs.get('size_mb', float('inf')) <= MODEL_SIZE_CAP_MB]
    ranked = sorted(under_cap, key=lambda t: t.value, reverse=True)
    print(f"\n✓ {prefix}: {len(completed)} completed, {len(study.trials) - len(completed)} pruned, "
          f"{len(completed) - len(under_cap)} over the {MODEL_SIZE_CAP_MB} MB cap ({(time.time() - t0) / 60:.1f} min)")
    print(f"  top {TOP_N} by score = {metric} - {SIZE_PENALTY_ALPHA} * size_mb:")
    for i, t in enumerate(ranked[:TOP_N]):
        print(f"    {i + 1}. {t.user_attrs.get('model_file'):<26} score {t.value:.4f}  pAUC {t.user_attrs.get('pauc', np.nan):.4f}  "
              f"AUC {t.user_attrs.get('auc', np.nan):.4f}  {t.user_attrs.get('size_mb', np.nan):6.2f} MB")
    return [{'model_file': t.user_attrs.get('model_file'), 'score': t.value, 'pauc': t.user_attrs.get('pauc'),
             'auc': t.user_attrs.get('auc'), 'size_mb': t.user_attrs.get('size_mb')} for t in ranked]

In [4]:
# ============================================================================
# SIMULATION ENGINE — optuna_strategy_nopenalty.ipynb (no trades-per-coin
# penalty), with these additions, all OFF unless an experiment turns them on:
#   * confidence threshold can be FIXED (FIXED_PARAMS)        -> experiment 3
#   * SCORE_MODE 'coinfrac' = capped mean x coin fraction     -> experiment 4
#   * exit_mode 'stop' = stale / loss floor SELL              -> experiment 5
#   * take-profit tuned when TUNE_TAKE_PROFIT (else static)
#   * each trade records its exit reason; compute_score reports where the
#     P&L comes from (take-profit vs end-of-data vs stop exits)
# ============================================================================
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path
import optuna
import gc
from numba import njit

# ========================================================================
# HELPER FUNCTION TO FORMAT DURATION
# ========================================================================

def format_duration(start_time, end_time):
    duration = end_time - start_time
    total_days = duration.total_seconds() / 86400
    months = int(total_days // 30)
    remaining_days = total_days - (months * 30)
    days = int(remaining_days)
    hours = int((remaining_days - days) * 24)
    parts = []
    if months > 0:
        parts.append(f"{months}mo")
    if days > 0 or months > 0:
        parts.append(f"{days}d")
    parts.append(f"{hours}h")
    return {
        'months': months, 'days': days, 'hours': hours,
        'total_days': total_days, 'formatted': " ".join(parts)
    }

# ========================================================================
# VECTORIZED PRE-COMPUTATION
# ========================================================================

def precompute_arrays(merged_df):
    n = len(merged_df)
    close_arr = merged_df['close'].values.astype(np.float64)

    return {
        'close': close_arr,
        'open': (merged_df['open'].values.astype(np.float64) if 'open' in merged_df else close_arr),
        'prediction': merged_df['prediction'].values.astype(np.float64),
        'liquidity': merged_df['estimated_sol_1pct_impact'].values.astype(np.float64),
        'price_position_15m': merged_df['price_position_15m'].values.astype(np.float64),
        'price_position_60m': merged_df['price_position_60m'].values.astype(np.float64),
        'n_rows': n,
    }

# ========================================================================
# CANDIDATE FINDER
# ------------------------------------------------------------------------
# A row fires as an entry candidate if:
#   prediction[i] >= confidence_threshold
#   AND price_position_15m[i] <= max_price_position_15m
#   AND price_position_60m[i] <= max_price_position_60m
# ========================================================================

@njit
def find_entry_candidates(prediction, price_position_15m, price_position_60m, n_rows,
                          conf_thresh, max_pp_15m, max_pp_60m):

    candidates = np.empty(n_rows, dtype=np.int64)
    count = 0

    for i in range(n_rows):
        if (prediction[i] >= conf_thresh and
                price_position_15m[i] <= max_pp_15m and
                price_position_60m[i] <= max_pp_60m):
            candidates[count] = i
            count += 1

    return candidates[:count]


# ========================================================================
# PRE-MERGE PREDICTIONS INTO OHLCV DATA
# ========================================================================

def precompute_merged_data(ohlcv_data_cache, test_df, xgb_model, feature_cols):
    """Pre-merge buy-model predictions into each coin's OHLCV timeline."""
    merged_cache = {}

    for coin_name, ohlcv_df in ohlcv_data_cache.items():
        coin_test = test_df[test_df['coin_id'] == coin_name].copy().reset_index(drop=True)
        if len(coin_test) == 0:
            continue

        X_test = coin_test[feature_cols].copy()
        X_test = X_test.loc[:, ~X_test.columns.duplicated()]
        X_test = X_test.apply(pd.to_numeric, errors='coerce')
        dtest = xgb.DMatrix(X_test, feature_names=feature_cols)
        predictions = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration))
        coin_test['prediction'] = predictions

        merged_df = ohlcv_df.merge(
            coin_test[['timestamp', 'prediction', 'estimated_sol_1pct_impact',
                        'price_position_15m', 'price_position_60m']],
            on='timestamp', how='left'
        )
        merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

        merged_df['prediction'] = merged_df['prediction'].fillna(0)
        merged_df['estimated_sol_1pct_impact'] = merged_df['estimated_sol_1pct_impact'].fillna(0)
        merged_df['price_position_15m'] = merged_df['price_position_15m'].fillna(1.0)
        merged_df['price_position_60m'] = merged_df['price_position_60m'].fillna(1.0)

        sim_start = merged_df['timestamp'].iloc[0]
        sim_end = merged_df['timestamp'].iloc[-1]
        duration_info = format_duration(sim_start, sim_end)

        arrays = precompute_arrays(merged_df)

        merged_cache[coin_name] = {
            'arrays': arrays,
            'duration_info': duration_info,
        }

    return merged_cache

# ========================================================================
# WORST-CASE SCORE SENTINEL
# ------------------------------------------------------------------------
# Returned by degenerate trials (no simulable coins). The score is mean capped
# final capital in dollars, so $0 — every coin wiped out — is genuinely the floor.
# ========================================================================

WORST_SCORE = 0.0

# ========================================================================
# REASON CODES FOR NUMBA
# ========================================================================

REASON_ACTIVATED = 0
REASON_END_OF_DATA = 2
REASON_STALE_EXIT = 3
REASON_REACHED_BOUND = 4
REASON_LOSS_FLOOR_ARMED = 5
REASON_LOSS_FLOOR_RECOVERY = 6
REASON_SIGNAL_EXIT = 7
REASON_TIME_EXIT = 8

# ========================================================================
# FAST TRADE SIMULATION
# ========================================================================

@njit
def trail_exit(arr_close, start_idx, n_rows, trail_pct, max_minutes):
    """Trailing phase after the take-profit is reached at start_idx: follow the
    running high of the close and exit at the first minute the close is
    trail_pct below it, after max_minutes, or at the end of the data.
    max_minutes = 0 exits at start_idx (= the plain take-profit)."""
    peak = arr_close[start_idx]
    end = min(n_rows - 1, start_idx + max_minutes)
    for j in range(start_idx + 1, end + 1):
        c = arr_close[j]
        if c > peak:
            peak = c
        if c <= peak * (1.0 - trail_pct):
            return j
    return end


@njit
def hold_exit(arr_close, score, start_idx, n_rows, h, max_minutes, trail_pct):
    """SIGNAL HOLD after the take-profit is reached at start_idx: keep holding while
    score >= h; exit at the first minute the score drops below h, the close falls
    trail_pct below its running high, max_minutes have passed, or the data ends.
    If the score is already below h at start_idx: exit there (= the take-profit)."""
    if score[start_idx] < h:
        return start_idx
    peak = arr_close[start_idx]
    end = min(n_rows - 1, start_idx + max_minutes)
    for j in range(start_idx + 1, end + 1):
        c = arr_close[j]
        if c > peak:
            peak = c
        if score[j] < h or c <= peak * (1.0 - trail_pct):
            return j
    return end


@njit
def signal_exit_idx(dir_s, move_s, start, stop, x_dir, z_move):
    """first minute in [start, stop) where the direction score < x_dir or the move
    score < z_move; -1 if none."""
    for j in range(start, stop):
        if dir_s[j] < x_dir or move_s[j] < z_move:
            return j
    return -1


@njit
def simulate_trade(arr_close, n_rows,
                   entry_idx, entry_price,
                   act_thresh, stale_minutes, loss_floor_pct,
                   start_idx, max_exit_idx, stale_active,
                   loss_floor_mode, n_consec_required,
                   max_price_seen_init):
    """Run from start_idx up to (but excluding) min(n_rows, max_exit_idx).

    Returns (exit_idx, reason, max_price_seen_final). The final max_price
    is returned so callers can resume the trade across multiple phase calls
    without losing the running high since entry.

    Priority at each minute: activation > loss-floor > stale.

    `stale_active=0` disables the stale check (used during the post-stale
    race phase where stale has already been crossed once).

    `loss_floor_mode`:
      0 — ignore the loss floor entirely (used in stale-race scan).
      1 — arm-watch (Phase 1): count consecutive minutes where
          arr_close[i] <= entry_price*(1 - floor). When the count reaches
          n_consec_required, return REASON_LOSS_FLOOR_ARMED. A minute
          above the floor resets the count.
      2 — recovery-watch (during loss-floor race): the FIRST minute where
          arr_close[i] > entry_price*(1 - floor) returns
          REASON_LOSS_FLOOR_RECOVERY.

    `max_exit_idx`: if the loop reaches this index without activating,
    return REASON_REACHED_BOUND. Pass n_rows for no bound.

    `max_price_seen_init`: pass -1.0 for fresh start (any close above -1.0
    becomes the new max). Pass the previously-returned max when resuming.
    """
    end = max_exit_idx if max_exit_idx < n_rows else n_rows
    if end <= start_idx:
        if end == n_rows:
            return n_rows - 1, REASON_END_OF_DATA, max_price_seen_init
        return start_idx - 1, REASON_REACHED_BOUND, max_price_seen_init

    max_price_seen = max_price_seen_init
    below_floor_count = 0

    for i in range(start_idx, end):
        minutes_held = i - entry_idx
        c_close = arr_close[i]

        if c_close > max_price_seen:
            max_price_seen = c_close

        # 1) Sell-threshold (highest priority)
        if max_price_seen >= entry_price * (1.0 + act_thresh):
            return i, REASON_ACTIVATED, max_price_seen

        # 2) Loss-floor handling
        if loss_floor_mode != 0:
            floor_price = entry_price * (1.0 - loss_floor_pct)

            if loss_floor_mode == 1:
                if c_close <= floor_price:
                    below_floor_count += 1
                    if below_floor_count >= n_consec_required:
                        return i, REASON_LOSS_FLOOR_ARMED, max_price_seen
                else:
                    below_floor_count = 0
            else:  # loss_floor_mode == 2: recovery-watch
                if c_close > floor_price:
                    return i, REASON_LOSS_FLOOR_RECOVERY, max_price_seen

        # 3) Stale (lowest priority)
        if stale_active:
            if minutes_held > stale_minutes:
                return i, REASON_STALE_EXIT, max_price_seen

    if end == n_rows:
        return n_rows - 1, REASON_END_OF_DATA, max_price_seen
    return end - 1, REASON_REACHED_BOUND, max_price_seen

# ========================================================================
# SIMULATION FUNCTION
# ========================================================================

def simulate_coin(coin_name, config, merged_cache):
    if coin_name not in merged_cache:
        return None

    cached = merged_cache[coin_name]
    arrays = cached['arrays']
    duration_info = cached['duration_info']

    candidates = find_entry_candidates(
        arrays['prediction'], arrays['price_position_15m'], arrays['price_position_60m'],
        arrays['n_rows'],
        config['confidence_threshold'],
        config['max_price_position_15m'],
        config['max_price_position_60m']
    )

    if len(candidates) == 0:
        return {
            'coin': coin_name, 'n_trades': 0,
            'n_reentries': 0,
            'n_original_wins': 0, 'n_original_losses': 0,
            'n_reentry_wins': 0, 'n_reentry_losses': 0,
            'n_loss_floor_arms': 0, 'n_loss_floor_recoveries': 0,
            'final_capital': config['initial_capital_usd'],
            'pnl_usd': 0, 'pnl_pct': 0, 'total_fees_usd': 0,
            'duration_days': duration_info['total_days'],
            'trades_per_day': 0,
            'trades': [],
        }

    capital = config['initial_capital_usd']
    total_fees = 0.0
    trades = []

    arr_close = arrays['close']
    arr_liquidity = arrays['liquidity']
    n_rows = arrays['n_rows']
    # EXECUTION CHECK (2026-11, off by default): 'fill_next' = 'open' / 'close' fills
    # EVERY entry and exit at the next minute's open / close instead of the signal
    # minute's close; 'se_fill_open' does that for signal exits only.
    arr_open = arrays.get('open', arr_close)
    _fill_next = str(config.get('fill_next', '') or '')
    _se_fill_open = float(config.get('se_fill_open', 0.0)) > 0.5

    slippage_pct = config['slippage_pct']
    fill_pct = config['target_fill_pct']

    # Strategy parameters.
    g_act_thresh = float(config['activation_threshold'])
    g_stale      = float(config['stale_trade_minutes'])
    g_reentry    = float(config['reentry_threshold_pct'])
    g_loss_floor = float(config['loss_floor_pct'])
    loss_floor_consec = int(config['loss_floor_consecutive_minutes'])

    desired_price_impact_pct = config['desired_price_impact_pct']
    sol_price_usd = config['sol_price_usd']
    initial_capital = config['initial_capital_usd']
    g_trading_start = int(config.get('trading_start_minutes', 0))   # no entries before this minute

    reentry_decay_rate_per_min = config['reentry_decay_rate']
    reentry_decay_start = float(config['reentry_decay_start_minutes'])
    # EXPERIMENT 5: exit_mode 'stop' SELLS when the stale timer or the loss floor
    # fires (instead of re-basing the entry and holding). 'rebase' = original.
    g_exit_mode_stop = (config.get('exit_mode', 'rebase') == 'stop')

    last_sell_close = -1.0
    last_sell_idx = -1

    # PER-TIER trade parameters (tier strategy, 2026-11): if the view carries a
    # per-minute 'tier_idx' and the config per-tier lists, every trade takes its
    # take-profit / stale / loss floor from the tier of its ENTRY minute (kept for
    # the trade's whole life, incl. rebases), and the re-entry threshold from the
    # tier of the candidate minute. Without them nothing changes.
    _tier_arr = arrays.get('tier_idx')
    _per_tier = _tier_arr is not None and 'tier_act' in config
    if _per_tier:
        _t_act = [float(x) for x in config['tier_act']]
        _t_stale = [float(x) for x in config['tier_stale']]
        _t_lf = [float(x) for x in config['tier_lf']]
        _t_re = [float(x) for x in config['tier_reentry']]

    # MODEL-DRIVEN STRATEGY OPTIONS (strategies, 2026-11); each is off unless its keys
    # are in the config. The view supplies per-minute 'combo_t', 'dir_t', 'move_t'.
    #   ts_*  take-profit / stale / loss floor scaled by the combo at entry (linear
    #         between combo 0.5 -> *_lo and combo 1.0 -> *_hi)
    #   se_*  signal exit: sell at the first minute (after se_min_hold) the direction
    #         score < se_dir or the move score < se_move; se_tp_off disables the take-profit
    #   sh_*  signal hold: at the take-profit keep holding while the combo >= sh_h
    #         (max sh_max minutes, or sh_trail below the running high)
    _ts_on = 'ts_tp_lo' in config
    if _ts_on:
        _ts = [float(config[k]) for k in ('ts_tp_lo', 'ts_tp_hi', 'ts_st_lo', 'ts_st_hi', 'ts_lf_lo', 'ts_lf_hi')]
    _se_on = 'se_dir' in config
    if _se_on:
        _se_dir, _se_move = float(config['se_dir']), float(config['se_move'])
        _se_min = int(config.get('se_min_hold', 0))
        _se_delay = int(config.get('se_delay', 0))     # diagnostic: execute the signal exit d minutes later
        if float(config.get('se_tp_off', 0.0)) > 0.5:
            g_act_thresh = 100.0
    _te_on = 'te_minutes' in config               # TIME EXIT: sell at market te_minutes after the entry
    _te_min = int(config.get('te_minutes', 0))
    _sh_on = 'sh_h' in config
    if _sh_on:
        _sh_h, _sh_max, _sh_trail = float(config['sh_h']), int(config['sh_max']), float(config['sh_trail'])
    _combo_t = arrays.get('combo_t')
    _dir_t = arrays.get('dir_t')
    _move_t = arrays.get('move_t')

    _trail_on = 'trail_pct' in config
    _trail_pct = float(config.get('trail_pct', 0.0))
    _trail_keep = float(config.get('trail_keep', 1.0))
    _trail_max = int(config.get('trail_max_minutes', 0))

    def passes_reentry_threshold(idx):
        if last_sell_close <= 0:
            return True
        re_thresh = _t_re[int(_tier_arr[idx])] if _per_tier else g_reentry
        minutes_since_sell = float(idx - last_sell_idx)
        if minutes_since_sell > reentry_decay_start:
            relaxation = (minutes_since_sell - reentry_decay_start) * reentry_decay_rate_per_min
            effective_reentry = re_thresh - relaxation
            if effective_reentry < -5.0:
                return True
            return arr_close[idx] <= last_sell_close * (1.0 - effective_reentry)
        else:
            return arr_close[idx] <= last_sell_close * (1.0 - re_thresh)

    c_ptr = 0
    n_candidates = len(candidates)

    while c_ptr < n_candidates:
        entry_idx = candidates[c_ptr]
        c_ptr += 1

        if entry_idx < g_trading_start:      # can't trade until TRADING_START_HOURS after listing
            continue
        if entry_idx <= last_sell_idx:
            continue
        if not passes_reentry_threshold(entry_idx):
            continue

        if _per_tier:
            _ti = int(_tier_arr[entry_idx])
            g_act_thresh, g_stale, g_loss_floor = _t_act[_ti], _t_stale[_ti], _t_lf[_ti]
        if _ts_on:
            _f = min(max((_combo_t[entry_idx] - 0.5) / 0.5, 0.0), 1.0)
            g_act_thresh = _ts[0] + (_ts[1] - _ts[0]) * _f
            g_stale = _ts[2] + (_ts[3] - _ts[2]) * _f
            g_loss_floor = _ts[4] + (_ts[5] - _ts[4]) * _f
        if _fill_next and entry_idx + 1 < n_rows:
            entry_idx = entry_idx + 1
            entry_price = (arr_open[entry_idx] if _fill_next == 'open' else arr_close[entry_idx]) * (1.0 + slippage_pct)
        else:
            entry_price = arr_close[entry_idx] * (1.0 + slippage_pct)
        available_capital = capital

        liq = arr_liquidity[entry_idx]
        if not np.isnan(liq) and liq > 0:
            sol_needed = liq * (desired_price_impact_pct * 100)
            max_usd = sol_needed * sol_price_usd
            amount = min(available_capital, max_usd)
            cap_active = True
        else:
            max_usd = np.inf                 # no liquidity value -> cap NOT applied (full capital)
            amount = available_capital
            cap_active = False
        cap_bound = cap_active and (max_usd < available_capital - 1e-9)   # did the cap actually limit the buy?

        if amount <= 0:
            continue

        shares = amount / entry_price
        capital -= amount

        buy_fee = shares * arr_close[entry_idx] * slippage_pct
        total_fees += buy_fee

        original_entry_price = entry_price
        original_entry_idx = entry_idx

        current_entry_idx = entry_idx
        current_entry_price = entry_price
        n_reentries = 0
        n_loss_floor_arms_this_trade = 0
        n_loss_floor_recoveries_this_trade = 0
        last_reentry_start_price = 0.0  # only used when n_reentries > 0
        final_reason_code = REASON_END_OF_DATA
        final_exit_idx = n_rows - 1

        # Per-segment state. phase1_start moves forward after a loss-floor
        # recovery so we don't re-scan minutes that already armed/recovered.
        # max_price_carry preserves the running high across phase boundaries
        # within the same segment; it resets to -1.0 on each re-entry.
        phase1_start = current_entry_idx + 1
        max_price_carry = -1.0

        while True:
            # ── Phase 1 (NORMAL) ───────────────────────────────────────
            # Stale active + loss-floor arm-watch. Activation wins ties.
            exit_idx, reason_code, max_price_carry = simulate_trade(
                arr_close, n_rows,
                current_entry_idx, current_entry_price,
                g_act_thresh, g_stale, g_loss_floor,
                phase1_start, n_rows, 1,
                1, loss_floor_consec,
                max_price_carry
            )

            if reason_code == REASON_ACTIVATED or reason_code == REASON_END_OF_DATA:
                final_exit_idx = exit_idx
                final_reason_code = reason_code
                break

            if g_exit_mode_stop:
                # stale / loss floor fired -> SELL at this bar's close (real stop)
                final_exit_idx = exit_idx
                final_reason_code = reason_code
                break

            # Find next qualifying re-entry candidate past exit_idx.
            # Same scan logic for both stale-race and loss-floor race:
            # advance c_ptr past anything ≤ exit_idx or failing the
            # re-entry threshold (those are permanently consumed,
            # matching prior behavior). Stop at the first qualifier;
            # don't bump c_ptr past it unless we actually take it.
            next_re_idx = -1
            qualifier_c_ptr = -1
            scan_ptr = c_ptr
            while scan_ptr < n_candidates:
                cand = candidates[scan_ptr]
                if cand <= exit_idx:
                    scan_ptr += 1
                    continue
                if not passes_reentry_threshold(cand):
                    scan_ptr += 1
                    continue
                next_re_idx = cand
                qualifier_c_ptr = scan_ptr
                break
            c_ptr = scan_ptr
            race_bound = n_rows if next_re_idx == -1 else next_re_idx

            if reason_code == REASON_STALE_EXIT:
                # ── Stale race (terminal) ──────────────────────────────
                # Stale fires once and latches; race runs with stale off
                # and loss-floor ignored. Either sell wins (finalize),
                # the qualifier wins (re-enter), or end-of-data.
                race_exit_idx, race_reason, _ = simulate_trade(
                    arr_close, n_rows,
                    current_entry_idx, current_entry_price,
                    g_act_thresh, g_stale, g_loss_floor,
                    exit_idx + 1, race_bound, 0,
                    0, loss_floor_consec,
                    max_price_carry
                )
                if race_reason == REASON_ACTIVATED or race_reason == REASON_END_OF_DATA:
                    final_exit_idx = race_exit_idx
                    final_reason_code = race_reason
                    break
                # REASON_REACHED_BOUND — take the qualifier
                c_ptr = qualifier_c_ptr + 1
                current_entry_idx = next_re_idx
                current_entry_price = arr_close[next_re_idx]
                last_reentry_start_price = arr_close[next_re_idx]
                n_reentries += 1
                phase1_start = current_entry_idx + 1
                max_price_carry = -1.0
                continue

            # ── Loss-floor race (revertible) ───────────────────────────
            # Stale still ticks; recovery (1 minute above floor) disarms
            # and resumes Phase 1 from the recovery minute. Possible
            # outcomes:
            #   ACTIVATED       → finalize
            #   END_OF_DATA     → finalize
            #   REACHED_BOUND   → qualifier wins; re-enter
            #   RECOVERY        → resume Phase 1 from race_exit_idx+1
            #   STALE_EXIT      → stale dominates; run terminal stale
            #                     race from race_exit_idx onward
            n_loss_floor_arms_this_trade += 1
            race_exit_idx, race_reason, race_max_price = simulate_trade(
                arr_close, n_rows,
                current_entry_idx, current_entry_price,
                g_act_thresh, g_stale, g_loss_floor,
                exit_idx + 1, race_bound, 1,
                2, loss_floor_consec,
                max_price_carry
            )

            if race_reason == REASON_ACTIVATED or race_reason == REASON_END_OF_DATA:
                final_exit_idx = race_exit_idx
                final_reason_code = race_reason
                break

            if race_reason == REASON_REACHED_BOUND:
                c_ptr = qualifier_c_ptr + 1
                current_entry_idx = next_re_idx
                current_entry_price = arr_close[next_re_idx]
                last_reentry_start_price = arr_close[next_re_idx]
                n_reentries += 1
                phase1_start = current_entry_idx + 1
                max_price_carry = -1.0
                continue

            if race_reason == REASON_LOSS_FLOOR_RECOVERY:
                # Resume Phase 1 from the minute AFTER recovery. Keep the
                # same segment (no re-entry happened); preserve max_price.
                n_loss_floor_recoveries_this_trade += 1
                phase1_start = race_exit_idx + 1
                max_price_carry = race_max_price
                continue

            # race_reason == REASON_STALE_EXIT — stale wins during loss-floor
            # race. Run a terminal stale-race from this point. The qualifier
            # we already located stays valid (race_exit_idx < race_bound).
            sr_exit_idx, sr_reason, _ = simulate_trade(
                arr_close, n_rows,
                current_entry_idx, current_entry_price,
                g_act_thresh, g_stale, g_loss_floor,
                race_exit_idx + 1, race_bound, 0,
                0, loss_floor_consec,
                race_max_price
            )
            if sr_reason == REASON_ACTIVATED or sr_reason == REASON_END_OF_DATA:
                final_exit_idx = sr_exit_idx
                final_reason_code = sr_reason
                break
            # REASON_REACHED_BOUND
            c_ptr = qualifier_c_ptr + 1
            current_entry_idx = next_re_idx
            current_entry_price = arr_close[next_re_idx]
            last_reentry_start_price = arr_close[next_re_idx]
            n_reentries += 1
            phase1_start = current_entry_idx + 1
            max_price_carry = -1.0
            # continue loops back to Phase 1

        # ---- EXIT FILL --------------------------------------------------------
        # An ACTIVATED exit fires on the first bar whose close is >= the activation
        # target, and the fill is booked at that bar's close. This matches live
        # behaviour: the decision is taken once the bar has closed, using only that
        # bar's close, and the order goes out ~5s later. Drift over those 5s is not
        # modelled (add it with the price WebSocket).
        #
        # overshoot_ratio records how far past the target that close actually was, so
        # you can see how much of the PnL depends on prices a live fill may not get.
        # LET WINNERS RUN (2026-11): with 'trail_pct' in the config, a position whose
        # take-profit is reached is not sold there: a share (1 - trail_keep) is sold at
        # the take-profit close, the rest follows the price and is sold trail_pct below
        # its running high (or after trail_max_minutes / at the end of the data). The
        # position counts as closed at the trailing exit. Without 'trail_pct' nothing changes.
        _truncated = False
        if _te_on and original_entry_idx + _te_min < final_exit_idx:
            final_exit_idx = original_entry_idx + _te_min
            final_reason_code = REASON_TIME_EXIT
            _truncated = True
        if _se_on:                                    # a model sell signal before the planned exit wins
            _sx = signal_exit_idx(_dir_t, _move_t, original_entry_idx + 1 + _se_min, final_exit_idx, _se_dir, _se_move)
            if _sx >= 0 and _sx + _se_delay < final_exit_idx:
                final_exit_idx = min(_sx + _se_delay, n_rows - 1)
                final_reason_code = REASON_SIGNAL_EXIT
                _truncated = True
        if _sh_on and final_reason_code == REASON_ACTIVATED:
            final_exit_idx = hold_exit(arr_close, _combo_t, final_exit_idx, n_rows, _sh_h, _sh_max, _sh_trail)
        _tp_close = arr_close[final_exit_idx]
        _blend_close = -1.0
        if _trail_on and final_reason_code == REASON_ACTIVATED:
            _tx = trail_exit(arr_close, final_exit_idx, n_rows, _trail_pct, _trail_max)
            _blend_close = (1.0 - _trail_keep) * _tp_close + _trail_keep * arr_close[_tx]
            final_exit_idx = _tx
        exit_close = arr_close[final_exit_idx] if _blend_close < 0 else _blend_close
        if _blend_close < 0 and (_fill_next or (_se_fill_open and final_reason_code == REASON_SIGNAL_EXIT)):
            _k = min(final_exit_idx + 1, n_rows - 1)
            exit_close = arr_close[_k] if _fill_next == 'close' else arr_open[_k]
            final_exit_idx = _k
        # LOOK-AHEAD FIX (2026-11-15): the trade above was played out to its take-profit /
        # end of data BEFORE the signal / time exit moved its end earlier; that imaginary
        # continuation advanced the candidate pointer (rebase / race scans). Rewind it to
        # the first candidate after the REAL exit, so nothing of the continuation is used.
        # legacy_cptr = True reproduces the old (biased) behaviour, for the tests only.
        if _truncated and not bool(config.get('legacy_cptr', False)):
            c_ptr = int(np.searchsorted(candidates, final_exit_idx, side='right'))
        if final_reason_code == REASON_ACTIVATED:
            _target = current_entry_price * (1.0 + g_act_thresh)
            overshoot_ratio = (exit_close / _target) if _target > 0 else np.nan
        else:
            overshoot_ratio = np.nan

        pnl_pct_raw = (exit_close - original_entry_price) / original_entry_price

        if pnl_pct_raw >= 0.0:
            realized_pct = pnl_pct_raw * fill_pct
        else:
            realized_pct = pnl_pct_raw * (2.0 - fill_pct)

        actual_price = original_entry_price * (1.0 + realized_pct)
        exit_price = actual_price * (1.0 - slippage_pct)

        sell_fee = shares * actual_price * slippage_pct
        total_fees += sell_fee

        proceeds = shares * exit_price
        capital += proceeds

        pnl_pct = (exit_price - original_entry_price) / original_entry_price
        pnl_usd = shares * (exit_price - original_entry_price)

        # ---- Per-segment win/loss accounting -----------------------------
        # A "segment" is the original entry plus each re-entry. Any segment
        # that triggered a re-entry counts as a loss (regardless of price).
        # The FINAL segment is win/loss based on the realized exit price vs
        # that segment's own start price.
        if n_reentries == 0:
            # Original IS the final segment
            if exit_price > original_entry_price:
                seg_orig_win, seg_orig_loss = 1, 0
            else:
                seg_orig_win, seg_orig_loss = 0, 1
            seg_re_win, seg_re_loss = 0, 0
        else:
            # Original re-entered -> automatic loss
            seg_orig_win, seg_orig_loss = 0, 1
            # Re-entries: all but the last triggered another re-entry -> losses;
            # the last re-entry is win/loss based on realized exit vs its start.
            if exit_price > last_reentry_start_price:
                seg_re_win = 1
                seg_re_loss = n_reentries - 1
            else:
                seg_re_win = 0
                seg_re_loss = n_reentries

        trades.append({
            'entry_idx': original_entry_idx,
            'exit_idx': final_exit_idx,
            'pnl_pct': pnl_pct,
            'pnl_usd': pnl_usd,
            'minutes_held': final_exit_idx - original_entry_idx,
            'n_reentries': n_reentries,
            'n_loss_floor_arms': n_loss_floor_arms_this_trade,
            'n_loss_floor_recoveries': n_loss_floor_recoveries_this_trade,
            'original_win': seg_orig_win,
            'original_loss': seg_orig_loss,
            'activated': final_reason_code == REASON_ACTIVATED,
            'exit_reason': int(final_reason_code),
            'overshoot_ratio': overshoot_ratio,
            'reentry_wins': seg_re_win,
            'reentry_losses': seg_re_loss,
            # --- fill / liquidity-cap diagnostics (from the ORIGINAL entry) ---
            'liq': liq,
            'available_capital': available_capital,
            'max_usd': max_usd,
            'amount': amount,
            # fraction of capital actually deployed. capital_after = capital_before*(1 + fill_frac*pnl_pct),
            # so log(final/initial) == sum of log(1 + fill_frac*pnl_pct) over trades, exactly.
            'fill_frac': (amount / available_capital) if available_capital > 0 else 0.0,
            'cap_active': cap_active,
            'cap_bound': cap_bound,
        })

        last_sell_close = arr_close[final_exit_idx]
        last_sell_idx = final_exit_idx

    trades_per_day = len(trades) / duration_info['total_days'] if duration_info['total_days'] > 0 else 0

    # Per-coin aggregates for the new scoring
    n_reentries_total = sum(t['n_reentries'] for t in trades)
    n_orig_win = sum(t['original_win'] for t in trades)
    n_orig_loss = sum(t['original_loss'] for t in trades)
    n_re_win = sum(t['reentry_wins'] for t in trades)
    n_re_loss = sum(t['reentry_losses'] for t in trades)
    n_lf_arms_total = sum(t['n_loss_floor_arms'] for t in trades)
    n_lf_recoveries_total = sum(t['n_loss_floor_recoveries'] for t in trades)

    return {
        'coin': coin_name,
        'n_trades': len(trades),
        'n_reentries': n_reentries_total,
        'n_original_wins': n_orig_win,
        'n_original_losses': n_orig_loss,
        'n_reentry_wins': n_re_win,
        'n_reentry_losses': n_re_loss,
        'n_loss_floor_arms': n_lf_arms_total,
        'n_loss_floor_recoveries': n_lf_recoveries_total,
        'final_capital': capital,
        'pnl_usd': capital - initial_capital,
        'pnl_pct': (capital - initial_capital) / initial_capital * 100,
        'total_fees_usd': total_fees,
        'duration_days': duration_info['total_days'],
        'trades_per_day': trades_per_day,
        'trades': trades,
    }

# ========================================================================
# SCORE  (mean per-coin capped final capital, in dollars)
# ========================================================================

def compute_score(all_results):
    """
    score = mean( min(final_capital, ceiling) )        (NO trades-per-coin penalty)

    Every coin gets its own independent initial_capital_usd, so total P&L across a set is
    exactly n * (arithmetic mean of final capital - initial). The ARITHMETIC mean is therefore
    the quantity that translates to money; a geometric mean would only be right if one account
    compounded through the coins sequentially. Per-coin downside is also bounded — a coin can
    cost you at most its allocation — so a wipeout is worth exactly -initial, no worse.

    The mean is winsorised at SCORE_CAPITAL_CEILING_X x starting capital first. That is a pure
    bias-for-variance trade: the raw arithmetic mean is the right objective but a dreadful
    ESTIMATOR on ~70 coins, because the top 1-3 coins can be 60-99% of total P&L and which coins
    those are does not transfer between splits. Winsorising biases the estimate down in exchange
    for one that generalises. Pick the ceiling deliberately — see SCORE_CAPITAL_CEILING_X.

    mean_log_return / geo_mean_x are still returned as diagnostics, they just no longer drive
    the score. The win rates are diagnostics only.

    Baseline is the starting capital, NOT zero: a coin that never trades scores exactly
    initial_capital_usd, so abstaining is worth $initial and still beats any losing config.
    NO-PENALTY VARIANT: nothing stops the optimiser from buying that. Abstention scores
    exactly $initial here. trades_per_coin is still computed and reported (diagnostic only),
    and 'tpc_penalty' is always 0.0 so downstream tables keep their columns.
    Returns (score, diagnostics).
    """
    if not all_results:
        return WORST_SCORE, {'median_final_capital': 0.0, 'avg_final_capital': 0.0, 'avg_capped_capital': 0.0,
                     'win_rate_originals': 1.0, 'win_rate_reentries': 1.0,
                     'total_trades': 0,
                     'n_unprofitable': 0,
                     'pct_profitable': 0.0, 'pct_untraded': 0.0, 'pct_unprofitable': 0.0,
                     'median_tpc': 0.0, 'n_scored_coins': 0,
                     'mean_log_return': WORST_SCORE, 'median_log_return': WORST_SCORE,
                     'trades_per_coin': 0.0, 'tpc_penalty': 0.0, 'total_pnl_usd': 0.0,
                     'geo_mean_x': 0.0, 'n_log_floored': 0, 'n_log_ceiled': 0,
                     'rebased_mean_log': float('nan'), 'n_rebased_harvests': 0}

    initial = globals().get('base_config', {}).get('initial_capital_usd', 20)
    _ceil_x = globals().get('SCORE_CAPITAL_CEILING_X', 10)
    ceiling = (_ceil_x * initial) if _ceil_x else float('inf')   # None/0 disables the ceiling
    log_floor_usd = globals().get('SCORE_LOG_FLOOR_X', 0.001) * initial

    caps = [r['final_capital'] for r in all_results]

    # ---- THE SCORE: mean per-coin log growth factor, clipped both ends ----
    # Floor/ceiling are for the log DIAGNOSTICS only; the score uses avg_capped below.
    log_rets = [np.log(min(max(c, log_floor_usd), ceiling) / initial) for c in caps]
    mean_log = float(np.mean(log_rets))
    median_log = float(np.median(log_rets))
    geo_mean_x = float(np.exp(mean_log))
    n_log_floored = int(sum(1 for c in caps if c < log_floor_usd))
    n_log_ceiled = int(sum(1 for c in caps if c > ceiling))

    # ---- RE-BASED LEG -----------------------------------------------------
    # A harvest (ACTIVATED exit) whose trade re-based at least once. pnl_pct is measured
    # vs the ORIGINAL entry while activation fires off the RE-BASED entry, so these book
    # losses. capital_after = capital_before*(1 + fill_frac*pnl_pct), so log1p(fill*pnl)
    # is each trade's exact contribution — the mean is what the leg costs per trade.
    # Log, not the arithmetic mean of pnl_pct: the -90%+ tail is what drives compounding
    # and an arithmetic mean under-weights it.
    _rebased = [t for r in all_results for t in r['trades']
                if t['activated'] and t['n_reentries'] > 0]
    n_rebased_harvests = len(_rebased)
    rebased_mean_log = (float(np.mean([np.log1p(t['fill_frac'] * t['pnl_pct']) for t in _rebased]))
                        if _rebased else float('nan'))   # NaN, not 0.0 — 0.0 would read as "no loss"

    # ---- capital ----------------------------------------------------------
    # avg_capped IS the score. avg_final_capital is the uncapped arithmetic mean, i.e.
    # true P&L per coin + initial — compare the two to see how much the ceiling is
    # discarding on each set.
    capped = [min(c, ceiling) for c in caps]
    avg_capped = float(np.mean(capped))
    avg_final_capital = float(np.mean(caps))
    median_final_capital = float(np.median(caps))
    total_pnl_usd = float(sum(c - initial for c in caps))

    ow = sum(r['n_original_wins'] for r in all_results)
    ol = sum(r['n_original_losses'] for r in all_results)
    rw = sum(r['n_reentry_wins'] for r in all_results)
    rl = sum(r['n_reentry_losses'] for r in all_results)
    wr_orig = (ow / (ow + ol)) if (ow + ol) > 0 else 1.0
    wr_re   = (rw / (rw + rl)) if (rw + rl) > 0 else 1.0

    tpc_all = [r['n_trades'] for r in all_results]
    median_tpc = float(np.median(tpc_all)) if tpc_all else 0.0
    total_trades = int(sum(r['n_trades'] for r in all_results))

    # ---- NO ANTI-ABSTENTION PENALTY (this variant) -------------------------
    # The original notebook subtracted a trades-per-coin penalty here. It is removed:
    # the score is the plain capped mean. trades_per_coin is kept as a diagnostic so
    # the two variants can be compared on how much each one actually trades.
    trades_per_coin = total_trades / len(all_results) if all_results else 0.0
    tpc_penalty = 0.0                       # always zero in this variant

    score = avg_capped

    n = len(all_results)
    n_profitable = sum(1 for r in all_results if r.get('pnl_usd', 0.0) > 0)
    n_untraded = sum(1 for r in all_results if r['n_trades'] == 0)
    n_unprofitable = sum(1 for r in all_results if r['n_trades'] > 0 and r.get('pnl_usd', 0.0) <= 0)
    pct_profitable = 100.0 * n_profitable / n if n else 0.0
    pct_untraded = 100.0 * n_untraded / n if n else 0.0
    pct_unprofitable = 100.0 * n_unprofitable / n if n else 0.0

    # ---- EXPERIMENT 4: alternative score ------------------------------------
    # coin_frac = (profitable coins + 0.5 * untraded coins) / coins
    # SCORE_MODE 'capped'   -> score = avg_capped               (default)
    # SCORE_MODE 'coinfrac' -> score = avg_capped * coin_frac
    # Both are always reported.
    coin_frac = (n_profitable + 0.5 * n_untraded) / n if n else 0.0
    score_capped = avg_capped
    score_coinfrac = avg_capped * coin_frac
    # SCORE_MODE 'winrate' -> score = avg_capped x pooled win rate of ALL trade segments
    # (originals + re-entries); no trades -> win rate 1.0 (abstaining keeps its $20)
    win_rate_all = ((ow + rw) / (ow + ol + rw + rl)) if (ow + ol + rw + rl) > 0 else 1.0
    score_winrate = avg_capped * win_rate_all
    if globals().get('SCORE_MODE', 'capped') == 'coinfrac':
        score = score_coinfrac
    elif globals().get('SCORE_MODE', 'capped') == 'winrate':
        score = score_winrate

    # ---- exit attribution: where does the P&L come from? ---------------------
    _all_tr = [t for r in all_results for t in r['trades']]
    def _pnl(sel):
        return float(sum(t['pnl_usd'] for t in _all_tr if sel(t)))
    _is_tp = lambda t: t.get('exit_reason') == REASON_ACTIVATED
    _is_eod = lambda t: t.get('exit_reason') == REASON_END_OF_DATA
    _is_stop = lambda t: t.get('exit_reason') not in (REASON_ACTIVATED, REASON_END_OF_DATA)
    n_losing_coins_eod = sum(1 for r in all_results
                             if r['n_trades'] > 0 and r.get('pnl_usd', 0.0) <= 0
                             and r['trades'][-1].get('exit_reason') == REASON_END_OF_DATA)

    return score, {'median_final_capital': median_final_capital, 'avg_final_capital': avg_final_capital,
                   'avg_capped_capital': avg_capped,
                   'win_rate_originals': wr_orig, 'win_rate_reentries': wr_re,
                   'total_trades': total_trades,
                   'n_unprofitable': n_unprofitable,
                   'pct_profitable': pct_profitable, 'pct_untraded': pct_untraded,
                   'pct_unprofitable': pct_unprofitable,
                   'median_tpc': median_tpc, 'n_scored_coins': len(all_results),
                   'mean_log_return': mean_log, 'median_log_return': median_log,
                   'trades_per_coin': trades_per_coin, 'tpc_penalty': tpc_penalty,
                   'total_pnl_usd': total_pnl_usd,
                   'geo_mean_x': geo_mean_x, 'n_log_floored': n_log_floored,
                   'n_log_ceiled': n_log_ceiled,
                   'rebased_mean_log': rebased_mean_log, 'n_rebased_harvests': n_rebased_harvests,
                   'coin_frac': coin_frac, 'score_capped': score_capped, 'score_coinfrac': score_coinfrac,
                   'win_rate_all': win_rate_all, 'score_winrate': score_winrate,
                   'n_trades_tp': sum(1 for t in _all_tr if _is_tp(t)),
                   'n_trades_eod': sum(1 for t in _all_tr if _is_eod(t)),
                   'n_trades_stop': sum(1 for t in _all_tr if _is_stop(t)),
                   'pnl_usd_tp': _pnl(_is_tp), 'pnl_usd_eod': _pnl(_is_eod),
                   'pnl_usd_stop': _pnl(_is_stop),
                   'n_losing_coins': n_unprofitable, 'n_losing_coins_eod': n_losing_coins_eod}


# ========================================================================
# OPTUNA OBJECTIVE FUNCTION
# ========================================================================

def objective(trial, base_config, coins_to_simulate, merged_cache, best_score_tracker):
    config = base_config.copy()

    # ---- Strategy params ----------
    _fx = globals().get('FIXED_PARAMS') or {}
    if 'confidence_threshold' in _fx:        # EXPERIMENT 3: forced strict gate
        config['confidence_threshold'] = float(_fx['confidence_threshold'])
    else:
        config['confidence_threshold'] = trial.suggest_float('confidence_threshold', *CONF_BOUNDS)
    config['max_price_position_15m'] = trial.suggest_float('max_price_position_15m', 0.05, 0.95)
    config['max_price_position_60m'] = trial.suggest_float('max_price_position_60m', 0.05, 0.95)
    if TUNE_TAKE_PROFIT:
        config['activation_threshold'] = trial.suggest_float('activation_threshold', *ACTIVATION_BOUNDS)
    else:
        config['activation_threshold'] = ACTIVATION_THRESHOLD
    config['stale_trade_minutes']    = trial.suggest_int('stale_trade_minutes', 60, 1440)
    config['reentry_threshold_pct']  = trial.suggest_float('reentry_threshold_pct', 0.005, 0.20)
    config['loss_floor_pct']         = trial.suggest_float('loss_floor_pct', 0.15, 0.40)

    config['reentry_decay_rate']          = trial.suggest_float('reentry_decay_rate', 0.0005, 0.01)
    config['reentry_decay_start_minutes'] = trial.suggest_int('reentry_decay_start_minutes', 20, 180)
    config['loss_floor_consecutive_minutes'] = trial.suggest_int('loss_floor_consecutive_minutes', 2, 15)


    all_results = []
    for coin_name in coins_to_simulate:
        result = simulate_coin(coin_name, config, merged_cache)
        if result is not None:
            all_results.append(result)

    if len(all_results) == 0:
        return WORST_SCORE

    total_trades = sum(r['n_trades'] for r in all_results)
    total_reentries = sum(r['n_reentries'] for r in all_results)
    total_loss_floor_arms = sum(r['n_loss_floor_arms'] for r in all_results)
    total_loss_floor_recoveries = sum(r['n_loss_floor_recoveries'] for r in all_results)
    avg_final_capital = np.mean([r['final_capital'] for r in all_results])
    median_final_capital = np.median([r['final_capital'] for r in all_results])
    profitable_coins = sum(1 for r in all_results if r['pnl_usd'] > 0)
    untraded_coins = sum(1 for r in all_results if r['n_trades'] == 0)

    # Trend-filter activity across all base-passing entry opportunities.

    # ---- Pooled win rates -------------------------------------------------
    total_orig_wins = sum(r['n_original_wins'] for r in all_results)
    total_orig_losses = sum(r['n_original_losses'] for r in all_results)
    total_re_wins = sum(r['n_reentry_wins'] for r in all_results)
    total_re_losses = sum(r['n_reentry_losses'] for r in all_results)

    orig_decided = total_orig_wins + total_orig_losses
    re_decided = total_re_wins + total_re_losses

    # Defaults to 1.0 when no originals / no re-entries occurred at all
    win_rate_originals = (total_orig_wins / orig_decided) if orig_decided > 0 else 1.0
    win_rate_reentries = (total_re_wins / re_decided) if re_decided > 0 else 1.0

    trial.set_user_attr('total_trades', total_trades)
    trial.set_user_attr('total_reentries', total_reentries)
    trial.set_user_attr('total_loss_floor_arms', total_loss_floor_arms)
    trial.set_user_attr('total_loss_floor_recoveries', total_loss_floor_recoveries)
    trial.set_user_attr('profitable_coins', profitable_coins)
    trial.set_user_attr('untraded_coins', untraded_coins)
    trial.set_user_attr('avg_final_capital', avg_final_capital)
    trial.set_user_attr('median_final_capital', median_final_capital)
    trial.set_user_attr('win_rate_originals', win_rate_originals)
    trial.set_user_attr('win_rate_reentries', win_rate_reentries)
    trial.set_user_attr('total_original_wins', total_orig_wins)
    trial.set_user_attr('total_original_losses', total_orig_losses)
    trial.set_user_attr('total_reentry_wins', total_re_wins)
    trial.set_user_attr('total_reentry_losses', total_re_losses)

    score, score_diag = compute_score(all_results)
    trial.set_user_attr('mean_log_return', score_diag.get('mean_log_return', 0.0))
    trial.set_user_attr('n_log_floored', score_diag.get('n_log_floored', 0))
    trial.set_user_attr('n_log_ceiled', score_diag.get('n_log_ceiled', 0))
    trial.set_user_attr('trades_per_coin', score_diag.get('trades_per_coin', 0.0))
    trial.set_user_attr('tpc_penalty', score_diag.get('tpc_penalty', 0.0))
    trial.set_user_attr('rebased_mean_log', score_diag.get('rebased_mean_log', float('nan')))
    trial.set_user_attr('n_rebased_harvests', score_diag.get('n_rebased_harvests', 0))
    trial.set_user_attr('avg_capped_capital', score_diag.get('avg_capped_capital', 0.0))
    trial.set_user_attr('total_pnl_usd', score_diag.get('total_pnl_usd', 0.0))
    trial.set_user_attr('unprofitable_coins', score_diag.get('n_unprofitable', 0))
    trial.set_user_attr('pct_profitable', score_diag.get('pct_profitable', 0.0))
    trial.set_user_attr('pct_untraded', score_diag.get('pct_untraded', 0.0))
    trial.set_user_attr('pct_unprofitable', score_diag.get('pct_unprofitable', 0.0))
    trial.set_user_attr('median_trades_per_coin', score_diag.get('median_tpc', 0.0))
    trial.set_user_attr('n_scored_coins', score_diag.get('n_scored_coins', 0))

    if score > best_score_tracker[0]:
        best_score_tracker[0] = score

    return score

# ========================================================================
# WARM UP NUMBA
# ========================================================================

def warmup_numba():
    dummy_arr = np.ones(20, dtype=np.float64)
    find_entry_candidates(dummy_arr, dummy_arr, dummy_arr, 20, 0.5, 0.5, 0.5)
    # simulate_trade signature: arr_close, n_rows, entry_idx, entry_price,
    # act_thresh, stale_minutes, loss_floor_pct, start_idx, max_exit_idx,
    # stale_active, loss_floor_mode, n_consec_required, max_price_seen_init
    simulate_trade(dummy_arr, 20, 0, 1.0,
                   0.1, 5.0, 0.25,
                   1, 20, 1, 1, 3, -1.0)
    print("✓ Numba JIT warmup complete")

In [5]:
# ============================================================================
# SIM HELPERS — same runner as before (stores each coin's result per run)
# ============================================================================
def merged_from_scores(ohlcv_cache, frame, score_col):
    """precompute_merged_data, but with a given score as the prediction."""
    merged = {}
    for coin, ohlcv_df in ohlcv_cache.items():
        ct = frame[frame['coin_id'] == coin]
        if len(ct) == 0:
            continue
        ct = ct[['timestamp', score_col, 'estimated_sol_1pct_impact',
                 'price_position_15m', 'price_position_60m']].rename(columns={score_col: 'prediction'})
        m = ohlcv_df.merge(ct, on='timestamp', how='left')
        m = m.loc[:, ~m.columns.duplicated()]
        m['prediction'] = m['prediction'].fillna(0)
        m['estimated_sol_1pct_impact'] = m['estimated_sol_1pct_impact'].fillna(0)
        m['price_position_15m'] = m['price_position_15m'].fillna(1.0)
        m['price_position_60m'] = m['price_position_60m'].fillna(1.0)
        merged[coin] = {'arrays': precompute_arrays(m),
                        'duration_info': format_duration(m['timestamp'].iloc[0], m['timestamp'].iloc[-1])}
    return merged



def per_coin_capped(params, coins, cache, cfg):
    config = cfg.copy()
    config.update(params)
    config.update(FIXED_PARAMS)
    cap = SCORE_CAPITAL_CEILING_X * config['initial_capital_usd']
    out = {}
    for c in coins:
        r = simulate_coin(c, config, cache)
        if r is not None:
            out[c] = float(min(r['final_capital'], cap))
    return out


def null_within(cache, rng):
    """Shuffle each coin's scored predictions in time, within the coin."""
    out = {}
    for coin, e in cache.items():
        a = dict(e['arrays'])
        p = a['prediction'].copy()
        s = np.flatnonzero(p > 0)
        p[s] = p[rng.permutation(s)]
        a['prediction'] = p
        out[coin] = {'arrays': a, 'duration_info': e['duration_info']}
    return out


def evaluate_params(params, coins, cache, cfg):
    config = cfg.copy()
    config.update(params)
    config.update(FIXED_PARAMS)
    if not TUNE_TAKE_PROFIT:
        config['activation_threshold'] = ACTIVATION_THRESHOLD
    res = [r for r in (simulate_coin(c, config, cache) for c in coins) if r is not None]
    return compute_score(res)


def _clean(x):
    if isinstance(x, (np.floating, float)):
        return float(x)
    if isinstance(x, np.integer):
        return int(x)
    return x


def run_arm(path, c_tune, c_val, cfg_over, n_runs, label):
    """N seeded optimisations (tune -> val-selected), saved after every run."""
    cfg = dict(base_config, **cfg_over)
    rec = json.load(open(path)) if path.exists() else {'cfg_over': cfg_over, 'runs': []}
    have = {r['run_idx'] for r in rec['runs']}
    coins_tune = [c for c in COINS_TUNE if c in c_tune]
    coins_val = [c for c in COINS_VAL if c in c_val]
    for run_idx in range(n_runs):
        if run_idx in have:
            continue
        t0 = time.time()
        study = optuna.create_study(direction='maximize', storage=None,
                                    sampler=optuna.samplers.CmaEsSampler(
                                        seed=OPTUNA_BASE_SEED + run_idx,
                                        n_startup_trials=CMAES_STARTUP_TRIALS))
        st = {'best_val': None, 'best_trial': None, 'best_params': None, 'best_tune': None,
              'best_diag': {}, 'series': [], 'stale': 0}

        def _cb(study_, trial):
            if trial.value is None or trial.number < EARLY_STOP_WARMUP:
                return
            if (trial.number - EARLY_STOP_WARMUP) % VAL_EVAL_EVERY != 0:
                return
            vs, vd = evaluate_params(trial.params, coins_val, c_val, cfg)
            st['series'].append(vs)
            tol = (max(EARLY_STOP_TOL_MIN, EARLY_STOP_TOL_FRAC * float(np.std(st['series'], ddof=1)))
                   if len(st['series']) >= EARLY_STOP_TOL_MIN_SAMPLES else EARLY_STOP_TOL_MIN)
            if st['best_val'] is None or vs > st['best_val'] + tol:
                st.update(best_val=vs, best_trial=trial.number, best_params=dict(trial.params),
                          best_tune=float(trial.value), best_diag=dict(vd), stale=0)
            else:
                st['stale'] += 1
            if (trial.number + 1) >= N_TRIALS and st['stale'] >= EARLY_STOP_PATIENCE_SAMPLES:
                study_.stop()

        tracker = [float('-inf')]
        study.optimize(lambda t: objective(t, cfg, coins_tune, c_tune, tracker),
                       n_trials=MAX_TRIALS, n_jobs=1, show_progress_bar=False, callbacks=[_cb])
        if st['best_params'] is None:
            bt = study.best_trial
            vs, vd = evaluate_params(bt.params, coins_val, c_val, cfg)
            st.update(best_val=vs, best_trial=bt.number, best_params=dict(bt.params),
                      best_tune=bt.value, best_diag=vd)
        per_coin = per_coin_capped(st['best_params'], coins_val, c_val, cfg)
        r = {'run_idx': run_idx, 'val_score': float(st['best_val']), 'tune_score': float(st['best_tune']),
             'per_coin_val': per_coin,
             'selected_trial': int(st['best_trial']), 'n_trials': len(study.trials),
             'params': {k: _clean(x) for k, x in st['best_params'].items()},
             'val_diag': {k: _clean(x) for k, x in st['best_diag'].items()}}
        rec['runs'].append(r)
        rec['runs'].sort(key=lambda x: x['run_idx'])
        json.dump(rec, open(path, 'w'), indent=2)
        d, p = r['val_diag'], r['params']
        _sc = (f"VAL ${r['val_score']:.2f}  (win rate {d.get('win_rate_all', float('nan')) * 100:.1f}%)"
               if globals().get('SCORE_MODE', 'capped') == 'capped' else
               f"VAL score {r['val_score']:.2f}  (money ${d.get('score_capped', float('nan')):.2f} x win rate "
               f"{d.get('win_rate_all', float('nan')) * 100:.1f}%)")
        print(f"    {label} run {run_idx + 1}/{n_runs}: {_sc}  "
              f"conf={p.get('confidence_threshold', float('nan')):.3f} "
              f"TP={p.get('activation_threshold', ACTIVATION_THRESHOLD):.3f}  "
              f"trades/coin={d.get('trades_per_coin', 0):.1f} untr={d.get('pct_untraded', 0):.0f}%  "
              f"[{(time.time() - t0) / 60:.1f} min]", flush=True)
        del study
        gc.collect()
    return [r for r in rec['runs'] if r['run_idx'] < n_runs]



warmup_numba()
print("Sim helpers ready.")

✓ Numba JIT warmup complete
Sim helpers ready.


In [6]:
# ============================================================================
# SIM SETUP — tune / val_tune scores, the OHLCV timelines, helpers
# ============================================================================
# Model predictions on tune / val_tune are cached per model as pred_<stem>_<split>.npy.
# A sim's buy score is a tune percentile: of the move score alone (B1), or of the
# product move x direction (B2, A50).
S = {}
for sp in ('tune', 'val_tune'):
    fr = load_split(sp)
    out = fr[_KEYS + ['estimated_sol_1pct_impact', 'price_position_15m']].copy()
    out['price_position_60m'] = fr['price_position_1h'].to_numpy()
    S[sp] = out
    del fr
    gc.collect()
ohlcv_data_cache = {}
for coin_name in pd.concat([S[sp]['coin_id'] for sp in S]).unique():
    o = _read_ohlcv_raw(coin_name)
    if o is not None:
        for col in ['open', 'high', 'low', 'close']:
            o[col] = o[col] * base_config['sol_price_usd']
        ohlcv_data_cache[coin_name] = o
COINS_TUNE = S['tune']['coin_id'].unique().tolist()
COINS_VAL = S['val_tune']['coin_id'].unique().tolist()
mode = SIM_MODES[0]
cfg_over = {'exit_mode': mode}
INFO = {}                                       # model stem -> its Phase A record


def add_predictions(records):
    """predict tune / val_tune for these models (cached) and add them to S as raw_<stem>."""
    todo = [r for r in records if f"raw_{Path(r['model_file']).stem}" not in S['tune']]
    for r in records:
        INFO[Path(r['model_file']).stem] = r
    if not todo:
        return
    for sp in ('tune', 'val_tune'):
        fr = None
        for r in todo:
            stem = Path(r['model_file']).stem
            p = OUT_DIR / f"pred_{stem}_{sp}.npy"
            if not p.exists():
                fr = load_split(sp) if fr is None else fr
                np.save(p, predict(load_model(OUT_DIR / r['model_file']), fr))
            S[sp][f"raw_{stem}"] = np.load(p)
        del fr
        gc.collect()


def _n(p):
    return len(json.load(open(p))['runs']) if p.exists() else 0


def sim(ms, ds=None):
    """RUNS_Y strategy optimisations: the move model alone (ds=None) or the pair; returns the val scores."""
    tag = ms if ds is None else f"{ms}__{ds}"
    path = OUT_DIR / f"sim_{tag}__{mode}.json"
    if _n(path) < RUNS_Y:
        col = f"buy_{tag}"
        for sp in S:
            if ds is None:
                S[sp][col] = to_percentile(S[sp][f"raw_{ms}"], S['tune'][f"raw_{ms}"])
            else:
                S[sp][col] = to_percentile(S[sp][f"raw_{ms}"] * S[sp][f"raw_{ds}"],
                                           S['tune'][f"raw_{ms}"] * S['tune'][f"raw_{ds}"])
        ct = merged_from_scores(ohlcv_data_cache, S['tune'], col)
        cv = merged_from_scores(ohlcv_data_cache, S['val_tune'], col)
        print(f"\n[{mode}] {tag}")
        run_arm(path, ct, cv, cfg_over, RUNS_Y, f"{mode}/{tag}")
        for sp in S:
            S[sp].drop(columns=[col], inplace=True)
        del ct, cv
        gc.collect()
    return np.array([r['val_score'] for r in sorted(json.load(open(path))['runs'], key=lambda r: r['run_idx'])
                     if r['run_idx'] < RUNS_Y])


def stage_table(results, label, metric_key, metric_name):
    rows = []
    for stem, v in results.items():
        m = INFO[stem]
        rows.append({label: stem, 'mean val $': v.mean(), 'se': v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
                     metric_name: m[metric_key], 'size MB': m['size_mb'], 'runs': ', '.join(f"{x:.2f}" for x in v)})
    return pd.DataFrame(rows).sort_values('mean val $', ascending=False).reset_index(drop=True)


def pick(table, label, manual):
    if manual:
        return manual
    if PICK_RULE == 'smallest_within_2se':
        best = table.iloc[0]
        close = table[table['mean val $'] >= best['mean val $'] - 2 * best['se']]
        return close.sort_values('size MB').iloc[0][label]
    return table.iloc[0][label]


print(f"✓ sim setup: {len(COINS_TUNE)} tune coins, {len(COINS_VAL)} val_tune coins")

✓ sim setup: 102 tune coins, 102 val_tune coins


In [7]:
# ============================================================================
# A1 — the MOVE model search
# ============================================================================
y_tr_m = train_df[MOVE_LABEL].astype(int)
y_va_m = val_df[MOVE_LABEL].astype(int).to_numpy()
_spw_m = float((len(y_tr_m) - y_tr_m.sum()) / max(y_tr_m.sum(), 1))
_dtr = xgb.DMatrix(train_df[FEATS['move']], label=y_tr_m, feature_names=FEATS['move'])
_dva = xgb.DMatrix(val_df[FEATS['move']], label=y_va_m, feature_names=FEATS['move'])
print(f"  move: train {len(y_tr_m):,} rows ({100 * y_tr_m.mean():.2f}% positive), scale_pos_weight {_spw_m:.2f}")
RANKED = {'move': run_hpo('move', 'move', _dtr, _dva, y_va_m, _spw_m)}
del _dtr, _dva
gc.collect()

  move: train 988,626 rows (17.15% positive), scale_pos_weight 4.83
MOVE MODEL HYPERPARAMETER SEARCH (move; metric pauc)
  0 prior trial(s); running 60 more


  [move trial   0/60] COMPLETE pAUC=0.6157  AUC=0.7927  score=0.6151  size=   0.6MB  |  best score=0.6151
  [move trial   1/60] COMPLETE pAUC=0.6156  AUC=0.7927  score=0.6144  size=   1.2MB  |  best score=0.6151
  [move trial   2/60] COMPLETE pAUC=0.6149  AUC=0.7914  score=0.6149  size=   0.0MB  |  best score=0.6151
  [move trial   3/60] COMPLETE pAUC=0.6159  AUC=0.7933  score=0.6139  size=   2.0MB  |  best score=0.6151
  [move trial   4/60] COMPLETE pAUC=0.6161  AUC=0.7936  score=0.6122  size=   3.9MB  |  best score=0.6151
  [move trial   5/60] COMPLETE pAUC=0.6131  AUC=0.7883  score=0.6130  size=   0.1MB  |  best score=0.6151
  [move trial   6/60] COMPLETE pAUC=0.6154  AUC=0.7913  score=0.6150  size=   0.4MB  |  best score=0.6151
  [move trial   7/60] COMPLETE pAUC=0.6157  AUC=0.7937  score=0.6138  size=   1.9MB  |  best score=0.6151
  [move trial   8/60] COMPLETE pAUC=0.6145  AUC=0.7916  score=0.6144  size=   0.1MB  |  best score=0.6151
  [move trial   9/60] COMPLETE pAUC=0.6161  AU

1903

In [8]:
# ============================================================================
# B1 — each of the TOP_N move models ALONE in strategy sims -> the chosen move model
# ============================================================================
TOP_MOVE = RANKED['move'][:TOP_N]
add_predictions(TOP_MOVE)
_t0 = time.time()
print(f"STAGE B1 — {len(TOP_MOVE)} move models alone, {RUNS_Y} runs each")
B1 = {Path(m['model_file']).stem: sim(Path(m['model_file']).stem) for m in TOP_MOVE}
T1 = stage_table(B1, 'move model', 'pauc', 'pAUC')
CHOSEN_MOVE = pick(T1, 'move model', MOVE_CHOICE)
print(f"\n  -> chosen move model: {CHOSEN_MOVE}  (${B1[CHOSEN_MOVE].mean():.2f} alone; {(time.time() - _t0) / 3600:.2f} h)")

STAGE B1 — 10 move models alone, 10 runs each

[rebase] move_trial027
    rebase/move_trial027 run 1/10: VAL $46.74  (win rate 58.1%)  conf=0.697 TP=0.290  trades/coin=13.0 untr=2%  [0.6 min]
    rebase/move_trial027 run 2/10: VAL $48.06  (win rate 53.7%)  conf=0.741 TP=0.303  trades/coin=13.9 untr=2%  [0.5 min]
    rebase/move_trial027 run 3/10: VAL $47.71  (win rate 57.4%)  conf=0.844 TP=0.263  trades/coin=13.6 untr=4%  [0.5 min]
    rebase/move_trial027 run 4/10: VAL $45.99  (win rate 50.9%)  conf=0.176 TP=0.244  trades/coin=17.5 untr=1%  [0.6 min]
    rebase/move_trial027 run 5/10: VAL $49.53  (win rate 50.9%)  conf=0.542 TP=0.339  trades/coin=12.3 untr=1%  [0.5 min]
    rebase/move_trial027 run 6/10: VAL $46.34  (win rate 44.5%)  conf=0.209 TP=0.280  trades/coin=16.1 untr=1%  [0.6 min]
    rebase/move_trial027 run 7/10: VAL $48.56  (win rate 64.6%)  conf=0.824 TP=0.268  trades/coin=12.8 untr=3%  [0.6 min]
    rebase/move_trial027 run 8/10: VAL $45.03  (win rate 58.9%)  conf=0.666 

In [9]:
# ============================================================================
# A2 — the DIRECTION model search, on the CHOSEN move model's candidate minutes
# ============================================================================
# Candidates (A50): tradable (12 h after listing, both price positions <= 0.575),
# the chosen move model's score >= its own train median, outcome decided.
chosen_move = load_model(OUT_DIR / f"{CHOSEN_MOVE}.ubj")
pm_tr, pm_va = predict(chosen_move, train_df), predict(chosen_move, val_df)
_thr = float(np.quantile(pm_tr, 1 - CAND_FRAC))
c_tr = eligible_flags(train_df) & (pm_tr >= _thr) & train_df[DIR_LABEL].isin([1, 2]).to_numpy()
c_va = eligible_flags(val_df) & (pm_va >= _thr) & val_df[DIR_LABEL].isin([1, 2]).to_numpy()
a_tr, a_va = train_df[c_tr].reset_index(drop=True), val_df[c_va].reset_index(drop=True)
y_tr_d = (a_tr[DIR_LABEL] == 2).astype(int)
y_va_d = (a_va[DIR_LABEL] == 2).astype(int).to_numpy()
_spw_d = float((len(y_tr_d) - y_tr_d.sum()) / max(y_tr_d.sum(), 1))
print(f"  direction: candidates of {CHOSEN_MOVE} — train {len(a_tr):,}, val {len(a_va):,} "
      f"({100 * y_tr_d.mean():.1f}% up first), scale_pos_weight {_spw_d:.2f}")
_dtr = xgb.DMatrix(a_tr[FEATS['dir']], label=y_tr_d, feature_names=FEATS['dir'])
_dva = xgb.DMatrix(a_va[FEATS['dir']], label=y_va_d, feature_names=FEATS['dir'])
DIR_PREFIX = f"dir_for_{CHOSEN_MOVE}"
RANKED['dir'] = run_hpo('dir', DIR_PREFIX, _dtr, _dva, y_va_d, _spw_d)
del _dtr, _dva, a_tr, a_va, chosen_move
gc.collect()

  direction: candidates of move_trial057 — train 153,512, val 85,251 (58.5% up first), scale_pos_weight 0.71
DIR MODEL HYPERPARAMETER SEARCH (dir_for_move_trial057; metric auc)
  0 prior trial(s); running 60 more


  [dir_for_move_trial057 trial   0/60] COMPLETE pAUC=0.5147  AUC=0.5898  score=0.5897  size=   0.0MB  |  best score=0.5897
  [dir_for_move_trial057 trial   1/60] COMPLETE pAUC=0.5158  AUC=0.5869  score=0.5867  size=   0.2MB  |  best score=0.5897
  [dir_for_move_trial057 trial   2/60] COMPLETE pAUC=0.5133  AUC=0.5945  score=0.5945  size=   0.1MB  |  best score=0.5945
  [dir_for_move_trial057 trial   3/60] COMPLETE pAUC=0.5142  AUC=0.5919  score=0.5918  size=   0.1MB  |  best score=0.5945
  [dir_for_move_trial057 trial   4/60] COMPLETE pAUC=0.5141  AUC=0.5923  score=0.5916  size=   0.8MB  |  best score=0.5945
  [dir_for_move_trial057 trial   5/60] COMPLETE pAUC=0.5132  AUC=0.5771  score=0.5770  size=   0.1MB  |  best score=0.5945
  [dir_for_move_trial057 trial   6/60] COMPLETE pAUC=0.5183  AUC=0.5885  score=0.5885  size=   0.0MB  |  best score=0.5945
  [dir_for_move_trial057 trial   7/60] COMPLETE pAUC=0.5142  AUC=0.5902  score=0.5891  size=   1.2MB  |  best score=0.5945
  [dir_for_move_

0

In [10]:
# ============================================================================
# B2 — each of the TOP_N direction models WITH the chosen move model -> the chosen direction model
# ============================================================================
TOP_DIR = RANKED['dir'][:TOP_N]
add_predictions(TOP_DIR)
_t0 = time.time()
print(f"STAGE B2 — chosen move model {CHOSEN_MOVE} x {len(TOP_DIR)} direction models, {RUNS_Y} runs each")
B2 = {Path(d['model_file']).stem: sim(CHOSEN_MOVE, Path(d['model_file']).stem) for d in TOP_DIR}
T2 = stage_table(B2, 'direction model', 'auc', 'AUC')
CHOSEN_DIR = pick(T2, 'direction model', DIR_CHOICE)
print(f"\n  -> chosen direction model: {CHOSEN_DIR}  ({(time.time() - _t0) / 3600:.2f} h)")

STAGE B2 — chosen move model move_trial057 x 10 direction models, 10 runs each

[rebase] move_trial057__dir_for_move_trial057_trial045
    rebase/move_trial057__dir_for_move_trial057_trial045 run 1/10: VAL $53.52  (win rate 66.3%)  conf=0.910 TP=0.224  trades/coin=13.9 untr=3%  [0.5 min]
    rebase/move_trial057__dir_for_move_trial057_trial045 run 2/10: VAL $50.27  (win rate 55.5%)  conf=0.852 TP=0.267  trades/coin=14.5 untr=2%  [0.5 min]
    rebase/move_trial057__dir_for_move_trial057_trial045 run 3/10: VAL $49.41  (win rate 50.6%)  conf=0.713 TP=0.285  trades/coin=15.2 untr=2%  [0.4 min]
    rebase/move_trial057__dir_for_move_trial057_trial045 run 4/10: VAL $50.02  (win rate 64.9%)  conf=0.931 TP=0.284  trades/coin=10.4 untr=3%  [0.5 min]
    rebase/move_trial057__dir_for_move_trial057_trial045 run 5/10: VAL $47.72  (win rate 69.4%)  conf=0.938 TP=0.214  trades/coin=11.3 untr=3%  [0.5 min]
    rebase/move_trial057__dir_for_move_trial057_trial045 run 6/10: VAL $48.44  (win rate 66.5%)

In [11]:
# ============================================================================
# FINAL — both stages, the value of the direction model, and the chosen pair
# ============================================================================
# Mean over the RUNS_Y runs (each the val-selected parameter set of one strategy
# optimisation), its standard error, the model's search metric and size, every run.
# Models whose means differ by less than ~2 standard errors are not reliably
# different: then prefer the smaller one (PICK_RULE = 'smallest_within_2se', or set
# MOVE_CHOICE / DIR_CHOICE by hand).
# VALUE OF THE DIRECTION MODEL: each pair minus the chosen move model ALONE, run by run
# (run i used the same sampler seed in both) -> mean difference, paired t-test.
alone = B1[CHOSEN_MOVE]
for stem, v in B2.items():
    d = v - alone
    T2.loc[T2['direction model'] == stem, 'vs move alone'] = d.mean()
    T2.loc[T2['direction model'] == stem, 'p (paired)'] = sst.ttest_rel(v, alone).pvalue if np.std(d) > 0 else np.nan
with pd.option_context('display.width', 250, 'display.max_columns', 20, 'display.max_colwidth', 90,
                       'display.float_format', lambda x: f"{x:.3f}"):
    print("=" * 100 + "\n  STAGE B1 — move models ALONE\n" + "=" * 100)
    print(T1.to_string(index=True))
    print("\n" + "=" * 100 + f"\n  STAGE B2 — direction models WITH the chosen move model {CHOSEN_MOVE}\n" + "=" * 100)
    print(T2[['direction model', 'mean val $', 'se', 'vs move alone', 'p (paired)', 'AUC', 'size MB', 'runs']].to_string(index=True))
final = B2[CHOSEN_DIR]
d = final - alone
print("\n" + "=" * 100)
print(f"  CHOSEN PAIR: {CHOSEN_MOVE}.ubj + {CHOSEN_DIR}.ubj  (pick rule: {PICK_RULE}"
      f"{', manual' if MOVE_CHOICE or DIR_CHOICE else ''})")
print(f"    move model alone       ${alone.mean():.2f} ± {alone.std(ddof=1) / np.sqrt(len(alone)):.2f}")
print(f"    with the direction model ${final.mean():.2f} ± {final.std(ddof=1) / np.sqrt(len(final)):.2f}")
print(f"    VALUE OF THE DIRECTION MODEL: {d.mean():+.2f} per coin ({100 * d.mean() / alone.mean():+.1f}%), better in "
      f"{int((d > 0).sum())}/{len(d)} runs, paired p {sst.ttest_rel(final, alone).pvalue:.4f}")
print(f"    size {INFO[CHOSEN_MOVE]['size_mb']:.2f} + {INFO[CHOSEN_DIR]['size_mb']:.2f} MB")
_all = T2['vs move alone'].to_numpy(dtype=float)
print(f"    across ALL {len(_all)} direction models: median {np.median(_all):+.2f}, {int((_all > 0).sum())}/{len(_all)} better than "
      f"the move model alone  (the chosen pair is the best of {len(_all)}, so its gain is somewhat optimistic; the median is the "
      f"fairer view of what a direction model adds)")
print("=" * 100)
print(f"\n  Copy {CHOSEN_MOVE}.ubj / {CHOSEN_DIR}.ubj (+ their _params.json) from {OUT_DIR}/ to the strategy notebook's folder.")
print("  These are VALIDATION scores of a selection among models — the honest number comes from the test split,")
print("  evaluated once in the strategy notebook after everything is fixed.")

  STAGE B1 — move models ALONE
      move model  mean val $    se  pAUC  size MB                                                                  runs
0  move_trial057      48.318 0.477 0.616    0.360  48.93, 47.09, 51.05, 50.58, 47.21, 48.86, 47.32, 48.12, 47.14, 46.88
1  move_trial055      47.444 0.348 0.616    0.215  48.96, 47.43, 49.10, 45.72, 47.63, 48.02, 47.07, 47.76, 46.54, 46.21
2  move_trial028      47.344 0.631 0.616    0.316  51.12, 48.68, 49.08, 46.61, 48.08, 44.52, 47.48, 47.09, 45.37, 45.41
3  move_trial045      47.190 0.430 0.616    0.295  48.93, 46.38, 46.05, 45.01, 47.11, 46.78, 48.23, 49.37, 47.66, 46.39
4  move_trial027      47.121 0.423 0.616    0.292  46.74, 48.06, 47.71, 45.99, 49.53, 46.34, 48.56, 45.03, 46.42, 46.82
5  move_trial029      47.011 0.589 0.616    0.284  48.48, 48.09, 44.37, 47.27, 44.40, 45.93, 48.13, 45.50, 48.02, 49.91
6  move_trial043      46.896 0.416 0.616    0.167  47.83, 49.08, 44.52, 46.36, 46.02, 45.76, 48.11, 46.64, 47.21, 47.44
7  move_t